In [ ]:
import sys
import os
import re
from collections import defaultdict
import numpy as np
from matplotlib import pyplot as plt
import json
import wandb

sys.path.append(os.path.join(os.getcwd(), '../../../'))
from utils import plot_utils

In [ ]:
import importlib
importlib.reload(plot_utils)

In [ ]:
api = wandb.Api()
username = 'julian-zim-uni-vie'
project_names = [project.name for project in api.projects(entity=username)]

In [ ]:
projects = {
	'll00': {
		'ns': {
			'bom' : list(),
			'hgm' : list(),
		},
		'ss': {
			'bom' : list(),
			'hgm' : list(),
		}
	},
}

for name in project_names:
	if name.startswith('mouse '):
		model_x_variant, ll = name.split(' ')[1].split('-')[0:2]
		projects[ll][model_x_variant.split('_')[1]][model_x_variant.split('_')[0]].append(name)

In [ ]:
hyperparams = {
	'linear': {
		ll: {
			variant: {
				model: {
					**(
						{'vb_reg_lambda': -1} if variant == 'ss' else
						{'wb_reg_lambda': -1}
					),
					**(
						{'wm_reg_lambda': -1} if model == 'hgm' else {}
					),
				} for model in projects[ll][variant].keys()
			} for variant in projects[ll].keys()
		} for ll in projects.keys()
	},
	'geometric': {
		ll: {
			variant: {
				model: {
					**(
						{'vb_reg_lambda': -1} if variant == 'ss' else
						{'wb_reg_lambda': -1}
					),
					**(
						{'wm_reg_lambda': -1} if model == 'hgm' else {}
					),
				} for model in projects[ll][variant].keys()
			} for variant in projects[ll].keys()
		} for ll in projects.keys()
	}
}
hyperparams_verbose = {
	ll: {
		variant: {
			model: {
				dataset: {
					'mpe_loss': {
						'all': np.empty(shape=()),
					},
					**(
						{'vb_reg_lambda': {
							'all': np.empty(shape=()),
							'mean': {
								'linear': -1.,
								'geometric': -1.,
							},
						}} if variant == 'ss' else
						{'wb_reg_lambda': {
							'all': np.empty(shape=()),
							'mean': {
								'linear': -1.,
								'geometric': -1.,
							},
						}}
					),
					**(
						{'wm_reg_lambda': {
							'all': np.empty(shape=()),
							'mean': {
								'linear': -1.,
								'geometric': -1.,
							},
						}} if model == 'hgm' else {}
					),
				} for dataset in [' '.join(project.split(' ')[-2:]) for project in projects[ll][variant][model]]
			} for model in projects[ll][variant].keys()
		} for variant in projects[ll].keys()
	} for ll in projects.keys()
}

In [ ]:
percentile = 1
all = True

for ll in hyperparams_verbose.keys():
	for variant in hyperparams_verbose[ll].keys():
		for model in hyperparams_verbose[ll][variant].keys():
			for dataset in hyperparams_verbose[ll][variant][model]:
				project = [name for name in projects[ll][variant][model] if name.endswith(dataset)][0]

				print(f'getting runs for \'{project}\'...', end='')
				runs = api.runs(f'{username}/{project}')
				mpe_losses = np.array([run.summary.get('mpe_loss', float('inf')) for run in runs])
				print(' done.')

				print('sorting them...', end='')
				mpe_losses, sorted_runs = zip(*sorted([(loss, run) for loss, run in zip(mpe_losses, runs) if loss != float('inf')], key=lambda x: x[0]))
				print(' done.')

				print(f'processing mpe losses...', end='')
				hyperparams_verbose[ll][variant][model][dataset]['mpe_loss']['all'] = mpe_losses if all else mpe_losses[:max(1, int(percentile / 100 * len(mpe_losses)))]
				print(' done.')

				if variant == 'ss':
					print(f'processing vb_reg_lambdas...', end='')
					vb_reg_lambdas = np.array([run.config['vb_reg_lambda'] for run in sorted_runs])
					best_vb_reg_lambdas = vb_reg_lambdas[:max(1, int(percentile / 100 * len(vb_reg_lambdas)))]
					hyperparams_verbose[ll][variant][model][dataset]['vb_reg_lambda']['all'] = vb_reg_lambdas if all else best_vb_reg_lambdas
					hyperparams_verbose[ll][variant][model][dataset]['vb_reg_lambda']['mean']['linear'] = np.mean(best_vb_reg_lambdas)
					hyperparams_verbose[ll][variant][model][dataset]['vb_reg_lambda']['mean']['geometric'] = np.power(10, np.mean(np.log10(best_vb_reg_lambdas)))
					print(' done.')
				else:
					print(f'processing wb_reg_lambdas...', end='')
					wb_reg_lambdas = np.array([run.config['wb_reg_lambda'] for run in sorted_runs])
					best_wb_reg_lambdas = wb_reg_lambdas[:max(1, int(percentile / 100 * len(wb_reg_lambdas)))]
					hyperparams_verbose[ll][variant][model][dataset]['wb_reg_lambda']['all'] = wb_reg_lambdas if all else best_wb_reg_lambdas
					hyperparams_verbose[ll][variant][model][dataset]['wb_reg_lambda']['mean']['linear'] = np.mean(best_wb_reg_lambdas)
					hyperparams_verbose[ll][variant][model][dataset]['wb_reg_lambda']['mean']['geometric'] = np.power(10, np.mean(np.log10(best_wb_reg_lambdas)))
					print(' done.')

				if model == 'hgm':
					print(f'processing wm_reg_lambdas...', end='')
					wm_reg_lambdas = np.array([run.config['wm_reg_lambda'] for run in sorted_runs])
					best_wm_reg_lambdas = wm_reg_lambdas[:max(1, int(percentile / 100 * len(wm_reg_lambdas)))]
					hyperparams_verbose[ll][variant][model][dataset]['wm_reg_lambda']['all'] = wm_reg_lambdas if all else best_wm_reg_lambdas
					hyperparams_verbose[ll][variant][model][dataset]['wm_reg_lambda']['mean']['linear'] = np.mean(best_wm_reg_lambdas)
					hyperparams_verbose[ll][variant][model][dataset]['wm_reg_lambda']['mean']['geometric'] = np.power(10, np.mean(np.log10(best_wm_reg_lambdas)))
					print(' done.')

			print('averaging across projects...', end='')
			for mean_type in hyperparams.keys():
				for hyperparam in hyperparams[mean_type][ll][variant][model].keys():
					means = np.array([
						hyperparams_verbose[ll][variant][model][dataset][hyperparam]['mean'][mean_type] for dataset in hyperparams_verbose[ll][variant][model].keys()
					])

					hyperparams[mean_type][ll][variant][model][hyperparam] = np.power(10, np.mean(np.log10(means))).item()
					# hyperparams[mean_type][ll][variant][model][hyperparam] = np.power(10, np.max(np.log10(means))).item()
			print(' done.\n')

In [ ]:
for mean_type in hyperparams.keys():
	print(f'{mean_type}:')
	for ll in hyperparams[mean_type].keys():
		print(f'\t{ll}')
		for variant in hyperparams[mean_type][ll].keys():
			print(f'\t\t{variant}')
			for model in hyperparams[mean_type][ll][variant].keys():
				print(f'\t\t\t{model}')
				for hyperparam in hyperparams[mean_type][ll][variant][model].keys():
					_mean = hyperparams[mean_type][ll][variant][model][hyperparam]
					print(f'\t\t\t\t{hyperparam}: \t{_mean}')

In [ ]:
# with open('../hyperparams.json', 'r') as f:
#     hyperparams = json.load(f)

In [ ]:
save = False
plot_utils.plot_hyperparams(hyperparams, hyperparams_verbose, None,
							ll='ll00', histograms=False, space='logarithmic',							save_dirpath='../../out/plots/hyperparameters' if save else None)

In [ ]:
with open('../hyperparams.json', 'w') as f:
    json.dump(hyperparams, f)